# SQL for Data Platforms
## Part 13: Choosing Your Database

A landscape/decision note, not a ranking, and not a syntax reference (that's Part 12) — three
categories of database, and the question each one actually answers well.

## Category 1 — OLTP engines: PostgreSQL, MySQL, SQL Server, Oracle

**Answers:** "record this transaction correctly, right now, with many people writing at once."

Row-oriented storage, normalized schemas (Part 1's `customers`/`orders`/`order_items`), strong
transactional guarantees (ACID), fast single-row reads/writes. This is the database *behind* an
application — what a checkout flow writes to, not what a BI dashboard reads from directly at
scale.

- **PostgreSQL** — open source, extensible (it's *why* DuckDB and dbt both target it as a
  reference dialect), the closest thing to a modern default choice for a new OLTP system.
- **MySQL** — open source, extremely widely deployed, historically the default for web
  applications.
- **SQL Server** — Microsoft's OLTP engine; T-SQL (Part 11/12); pairs naturally with Microsoft
  Fabric on the analytics side.
- **Oracle** — the traditional enterprise-database default in large, older organizations; PL/SQL
  (Part 11) is Oracle-specific and still common in banking, telecom, and government systems.

## Category 2 — Cloud analytical warehouses: Snowflake, BigQuery, Databricks, Fabric, Redshift

**Answers:** "scan and aggregate billions of rows for analysis, cheaply, without me managing a
cluster."

Columnar storage, elastic/serverless compute, separated storage and compute (mostly — Redshift is
the exception, Part 12). This is what a BI tool, a data scientist's notebook, or a scheduled
transformation pipeline (Part 10's dbt) actually queries.

- **Snowflake** — pure SaaS warehouse, storage/compute fully separated, broad ecosystem support.
- **BigQuery** — Google Cloud's serverless warehouse, pay-per-bytes-scanned by default.
- **Databricks** — grew from Spark/data-lake roots; strong for combining SQL with heavier
  Python/ML workloads on the same platform.
- **Microsoft Fabric** — Microsoft's unified analytics platform (Warehouse + Lakehouse items on
  OneLake); newest of the five, verify current maturity of specific features against docs.
- **Redshift** — AWS's warehouse, Postgres-derived syntax, distribution-style architecture
  (Part 12) rather than fully separated storage/compute.

## Category 3 — Embedded / local: SQLite, DuckDB

**Answers:** "run real SQL with no server, for development, testing, or a genuinely small dataset
that will never need a cluster."

- **SQLite** — row-oriented, the engine behind this entire series' Parts 1–8; also the default
  embedded database inside countless applications and mobile apps.
- **DuckDB** — columnar, "SQLite for analytics" — Part 9's partition-pruning demo and Part 10's
  dbt-duckdb project both lean on this being a genuinely capable analytical engine, not just a toy.

Neither is a warehouse replacement at real production scale — but for local development, CI test
suites, or a dataset that fits on one machine, reaching for a cloud warehouse is often overkill.

## Decision framing

| Question | Points toward |
|---|---|
| Does this serve a live application's reads/writes? | An OLTP engine (Category 1) |
| Am I aggregating/analyzing large historical volumes? | A cloud warehouse (Category 2) |
| Am I developing/testing locally, or is the data genuinely small? | SQLite/DuckDB (Category 3) |
| Do I already have a dominant cloud provider? | Often decides the warehouse for you (BigQuery→GCP, Redshift→AWS, Fabric→Azure) — Snowflake and Databricks are the two credible multi-cloud choices |
| Does my organization already run Oracle/SQL Server for OLTP? | Worth knowing PL/SQL/T-SQL (Part 11) regardless of which warehouse sits alongside it |

None of this is permanent — many real architectures run more than one category at once (an OLTP
engine feeding a warehouse via CDC, per
[Orchestration & CDC](../../mfzamudio.github.io/publications/pattern-orchestration-cdc.html)).

## Best Practices — Choosing a Database

- Don't default to "whatever's newest" — an OLTP question answered with a warehouse (or vice
  versa) usually surfaces as a performance or cost problem 6 months later, not immediately.
- Existing cloud commitment is usually the strongest real-world constraint, often stronger than
  any feature comparison.
- Keep the three categories distinct in your own head even when a platform blurs them (Databricks
  and Fabric both increasingly do more than one job) — know which job a given query is actually
  doing.

## Next

**Part 14 — SQL Style & Best Practices** closes the series with the cross-cutting habits that
apply no matter which platform from this page you end up on.